In [ ]:
import os
import threading
from concurrent.futures import ThreadPoolExecutor

import dataiku


In [ ]:
client = dataiku.api_client()
code_envs = client.list_code_envs()
failed_builds = []
_lock = threading.Lock()


In [ ]:
def get_directory_size(path):
    """Return the total byte size of all non-symlink files under path."""
    total_size = 0
    for dirpath, _, filenames in os.walk(path, followlinks=False):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)
            try:
                if not os.path.islink(file_path):
                    total_size += os.path.getsize(file_path)
            except FileNotFoundError:
                continue
    return total_size


def human_readable_size(size_bytes):
    """Convert a byte count to a human-readable string (e.g., 1.50 GB)."""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB', 'PB']:
        if size_bytes < 1024:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.2f} EB"


def _process_code_env(code_env_info):
    """Rebuild one code environment from scratch and report failures."""
    # Each worker creates its own client to avoid sharing a single connection across threads.
    env_client = dataiku.api_client()
    env_name = code_env_info['envName']
    try:
        code_env = env_client.get_code_env(code_env_info['envLang'], env_name)
        print(f'Starting rebuild of {env_name} ...')
        # Update this path to match your DSS data directory (typically DSS_DATA_DIR/code-envs/python).
        env_path = os.path.join('/data/dataiku/dss_data/code-envs/python', env_name)

        res = code_env.update_packages(force_rebuild_env=True)

        if not res['messages']['success']:
            print(f"FAILED: {env_name}")
            with _lock:
                failed_builds.append(env_name)
            print(res)

    except Exception as exc:
        print(f"Exception rebuilding {env_name}: {exc}")


In [ ]:
max_workers = os.cpu_count() or 1
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    executor.map(_process_code_env, code_envs)

if failed_builds:
    print(f'Environments that failed to build: {failed_builds}')

print('Finished rebuilding all code environments from scratch')